# MaxFrame AI Function：百炼 VL 模型图片理解

## 场景描述

通过 MaxFrame AI Function 调用百炼平台多模态模型（`qwen3.6-plus`），对 OSS 上的店铺封面图进行分布式图片理解（图片描述生成）。

适用于：商品图片自动标注、内容审核、图像内容摘要等场景。

### 前置条件

- 已安装 MaxFrame SDK >= 2.6.1（`pip install --upgrade maxframe`）
- MaxCompute Project：含 `cg_mall_shops_pic_meta` meta 表
- OSS Bucket：`oss-yiwu-sft-qwen`（杭州 region，内网 endpoint `oss-cn-hangzhou-internal.aliyuncs.com`）
- 已开通模型计算 Quota 资源（Token 计费），如使用项目默认 Quota 可不显式指定


## 1. 环境准备

In [ ]:
import os

import pandas as pd
import maxframe.dataframe as md
from maxframe import new_session
from maxframe.config import options
from maxframe.learn.contrib.llm import ImageContentType
from maxframe.learn.utils import read_odps_model
from odps import ODPS
from odps import options as odps_options

odps_options.catalog.endpoint = f"http://{o.get_catalog_host()}"

## 2. 配置引擎与创建会话

百炼 VL 模型推理通过 DPE 引擎执行，按 Token 消耗计费

In [ ]:
# 配置执行引擎：DPE 优先
options.dag.settings = {
    "engine_order": ["DPE", "MCSQL"],
    "unavailable_engines": ["SPE"],
}

# 使用 Token Quota 计算资源（百炼商业化模型，按 Token 计费）
# 如未单独申请 Token Quota，可注释掉本行使用项目默认 Quota
# options.session.inference_quota_name = "<your-token-quota-name>"

# ---- ODPS / MaxCompute ----
ODPS_ACCESS_ID = "xxxxx"
ODPS_SECRET_ACCESS_KEY = "xxxxx"
ODPS_PROJECT = "xxxxx"             # 例如 postpayproject
ODPS_ENDPOINT = "xxxxx"            # 例如 http://service.cn-shanghai.maxcompute.aliyun.com/api
ODPS_CATALOG_ENDPOINT = "xxxxx"    # 例如 http://service.cn-shanghai.maxcompute.aliyun.com/

o = ODPS(
    access_id=ODPS_ACCESS_ID,
    secret_access_key=ODPS_SECRET_ACCESS_KEY,
    project=ODPS_PROJECT,
    endpoint=ODPS_ENDPOINT,
    catalog_endpoint=ODPS_CATALOG_ENDPOINT,
)

# 创建 MaxFrame 会话（使用 vl_new 主版本，支持 AI Function VL 模型）
session = new_session(o)
print(f"Session ID: {session.session_id}")
print(f"Logview: {session.get_logview_address()}")

## 3. 加载百炼 VL 模型

从 MaxCompute 公共模型集加载百炼预注册的多模态理解模型。

In [ ]:
# 列出百炼公共模型集中的所有可用模型
available_models = list(o.list_models(project="bigdata_public_modelset"))
print(f"可用模型数量: {len(available_models)}")
print("可用模型列表:")
for m in available_models:
    print(f"  - {m.name}")

# 从百炼公共模型集加载 VL 模型
image_llm = read_odps_model(
    "qwen3.6-plus", project="bigdata_public_modelset"
)


## 4. 读取店铺封面图 meta 表

参考 `shops.py`，从 `cg_mall_shops_pic_meta` 读取下载成功（`status='ok'`）的封面图记录，并把 `oss_path`（`oss://<bucket>/<path>` 形式）拼接上 endpoint，得到 AI Function 需要的 `oss://<endpoint>/<bucket>/<path>` 形式的 `image_url` 列。


In [ ]:
# ---- OSS 配置（杭州内网 endpoint，与 shops.py 一致） ----
OSS_INTERNAL_ENDPOINT = "oss-cn-hangzhou-internal.aliyuncs.com"
OSS_BUCKET = "xxxxx"
OSS_PICTURES_DIR = "pic_shops"
OSS_ACCESS_KEY_ID = "xxxxx"
OSS_ACCESS_KEY_SECRET = "xxxxx"

storage_options = {
    "access_key_id": OSS_ACCESS_KEY_ID,
    "access_key_secret": OSS_ACCESS_KEY_SECRET,
}

# ---- 店铺封面图 meta 表（由 shops.py 写入） ----
META_TABLE = "xxxxx"

df = md.read_odps_table(
    META_TABLE,
    columns=["shop_id", "pic_name", "oss_path", "status", "name"],
).mf.rebalance(num_partitions=20)

# 仅保留下载成功的封面图
df = df[df["status"] == "ok"]

# meta 表里 oss_path 形如 oss://<bucket>/<key>，需补上 endpoint 才能被 AI Function 读取
image_url = df["oss_path"].str.replace(
    "oss://", f"oss://{OSS_INTERNAL_ENDPOINT}/", regex=False
)
df = df.assign(image_url=image_url).rebalance(10)


## 5. 调用 VL 模型进行图片理解

通过 `generate()` 接口，将文本 Prompt 和 OSS 图片组合传入模型。

In [ ]:
cp = image_llm.content_part

result_df = image_llm.generate(
    df,
    messages=[
        {
            "role": "user",
            "content": [
                cp.text("请用一句话描述这张图片"),
                cp.image(
                    data=df.image_url,
                    type=ImageContentType.IMAGE_URL,
                    storage_options=storage_options,
                ),
            ],
        }
    ],
    simple_output=True,
    running_options={"total_rpm_limit": 50000},
    params={"max_tokens": 128},
)

## 6. 执行推理并查看结果

MaxFrame 采用懒执行模式，调用 `.execute()` 触发实际计算。

In [ ]:
result = result_df.execute()
print(result)

## 7. （可选）将结果写入 MaxCompute 表

In [ ]:
# 将推理结果写入 MaxCompute 表，供下游任务消费
# md.to_odps_table(result_df, "your_output_table", overwrite=True).execute()

## 8. 资源清理

In [ ]:
session.destroy()
print("会话已销毁")